In [ ]:
!pip install mne -q
!pip install -U "decorator>=5.1" scikit-learn google-genai -q

In [ ]:
import mne

# Verifica a versão instalada
print(f"Versão do MNE: {mne.__version__}")

print("Sucesso! O MNE está pronto para uso.")

In [ ]:
import mne

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from pathlib import Path

caminhos_edf = (Path.cwd() / "chb24_01.edf", Path.cwd().parent / "chb24_01.edf")
arquivo_edf = next((caminho for caminho in caminhos_edf if caminho.is_file()), None)
if arquivo_edf is None:
    raise FileNotFoundError("Não foi possível localizar chb24_01.edf na pasta do notebook ou na raiz do projeto.")

raw = mne.io.read_raw_edf(arquivo_edf, preload=True)
raw

In [ ]:
print(raw.info)
print(raw.ch_names)
print(f"Taxa de amostragem: {raw.info['sfreq']} Hz")

os códigos FP, F, P, T, C são códigos neuro encefalograma que significam:
F = Frontal
FP = frontal polar
P = parietal
T = temporal
C = hospital

In [ ]:
raw.plot(duration=10, n_channels=5)

In [ ]:
data_filtrado = raw.get_data()
data_filtrado.shape

In [ ]:
print("Número de canais:", data_filtrado.shape[0])
print("Número de amostras:", data_filtrado.shape[1])

In [ ]:
caminhos_resumo = (Path.cwd() / "chb24-summary.txt", Path.cwd().parent / "chb24-summary.txt")
arquivo_resumo = next((caminho for caminho in caminhos_resumo if caminho.is_file()), None)
if arquivo_resumo is None:
    raise FileNotFoundError("Não foi possível localizar chb24-summary.txt na pasta do notebook ou na raiz do projeto.")

with arquivo_resumo.open("r") as f:
  conteudo = f.readlines()

for linha in conteudo[:40]:
  print(linha.strip())

In [ ]:
seizure_intervals_sec = [
    (480, 505),
    (2451, 2476)
]

sfreq = raw.info["sfreq"]

Intervalos pra saber quando a crise acontece e o sfreq usado pra transformar o tempo em indice

In [ ]:
canal_idx = 0
signal = data_filtrado[canal_idx]

In [ ]:
time = np.arange(len(signal)) / sfreq

In [ ]:
plt.figure(figsize=(16,4))
plt.plot(time, signal, linewidth=0.7)

for i, (start, end) in enumerate(seizure_intervals_sec):
  plt.axvspan(start, end, color="red", alpha=0.25,
              label="Crise" if i==0 else None)

plt.xlabel("Tempo (s)")
plt.ylabel("Amplitude")
plt.title("EEG com intervalos de crise destacados")
plt.legend()
plt.show()

In [ ]:
window_size = 256
step = 128

windows = []

for start in range(0, data_filtrado.shape[1] - window_size, step):
  segment = data_filtrado[:, start:start + window_size]
  windows.append(segment)

windows = np.array(windows)
windows.shape

In [ ]:
plt.figure(figsize=(12,4))
plt.plot(windows[0][0])
plt.title("Ex de janela do Canal 1")
plt.show()

In [ ]:
print("Quantidade de janelas:", windows.shape[0])
print("Canais por janela:", windows.shape[1])
print("Pontos por janela:", windows.shape[2])

In [ ]:
features = []

for window in windows:
  feat = []
  feat.extend(window.mean(axis=1))
  feat.extend(window.std(axis=1))
  feat.extend(window.min(axis=1))
  feat.extend(window.max(axis=1))
  features.append(feat)

X = np.array(features)
X.shape

In [ ]:
# lista onde vamos guarda o rótulo de cada janela
labels = []

# percorre o sinal da mesma forma que criamos as janelas
for start in range(0, data_filtrado.shape[1] - window_size, step):
    end = start + window_size # fim da janela

    # converte indice -> tempo (segundos)
    start_sec = start / sfreq
    end_sec = end / sfreq

    label = 0 # assume que é normal

    # verifica se a janela encosta em algum intervalo de crise
    for seizure_start, seizure_end in seizure_intervals_sec:
        overlap = (start_sec < seizure_end) and (end_sec > seizure_start)

        if overlap:
            label = 1 # marca como crise
            break

    labels.append(label) # salva o rótulo da janela

# transforma em array e exibe o resumo uma única vez
y = np.array(labels)
print("Janelas totais:", len(y))
print("Janelas com crise:", y.sum())
print("Janelas normais:", (y == 0).sum())

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sklearn.tree import DecisionTreeClassifier

modelo_arvore = DecisionTreeClassifier(random_state=42)
modelo_arvore.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

# o modelo faz previsões em dados que ele nunca viu (teste)
y_pred = modelo_arvore.predict(X_test)

# comparando algumas previsões com o valor real
print("Comparação entre valor real e previsto:")
for i in range(10):
    print(f"Real: {y_test[i]} | Previsto: {y_pred[i]}")
        
# calculando a acurácia (quantos ele acertou no total)
acc = accuracy_score(y_test, y_pred)

print("\nAcurácia do modelo:", acc)

In [ ]:
import matplotlib.pyplot as plt

# Para visualizar o sinal inteiro, as previsões precisam seguir a ordem de X.
# y_pred contém apenas as previsões do subconjunto X_test.
y_pred_all = modelo_arvore.predict(X)
signal = data_filtrado[0]

# Une janelas anômalas consecutivas em uma única região para reduzir o custo
# de renderização e evitar centenas de faixas sobrepostas.
anomalous_windows = np.flatnonzero(y_pred_all == 1)
anomalous_regions = []

if anomalous_windows.size:
    region_start = region_end = anomalous_windows[0]
    for window_idx in anomalous_windows[1:]:
        if window_idx <= region_end + 1:
            region_end = window_idx
        else:
            anomalous_regions.append((region_start, region_end))
            region_start = region_end = window_idx
    anomalous_regions.append((region_start, region_end))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(signal, color="gray", alpha=0.6, linewidth=0.7)

for start_window, end_window in anomalous_regions:
    start = start_window * step
    end = end_window * step + window_size
    ax.axvspan(start, end, color="red", alpha=0.3)

ax.set_title("Sinal de EEG com regiões detectadas como atividade anômala")
ax.set_xlabel("Tempo (amostras)")
ax.set_ylabel("Amplitude")
fig.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
amostra = X_test[0].reshape(1, -1)
pred = modelo_arvore.predict(amostra)[0]
print("Predição da amostra:", pred)


In [ ]:
from sklearn.neural_network import MLPClassifier

modelo_mlp = MLPClassifier(
    hidden_layer_sizes=(64,32),
    max_iter=300,
    random_state=42
)

modelo_mlp.fit(X_train, y_train)

In [ ]:
y_pred_mlp = modelo_mlp.predict(X_test)

print("Comparação entre valor real e previsto (Rede Neural)")
for i in range(10):
    print(f"Real: {y_test[i]} | Previsto: {y_pred_mlp[i]}")

acc_mlp = accuracy_score(y_test, y_pred_mlp)

print("\nAcurácia da rede neural:", acc_mlp)

In [ ]:
print("Árvore de decisão:", acc)
print("Rede neural:", acc_mlp)

In [ ]:
X_d1 = windows.transpose(0, 2, 1) # amostras, tempo, canais
X_d1.shape

In [ ]:
indice = 0
amostra_features = X_test[indice].reshape(1, -1)
pred_final = modelo_mlp.predict(amostra_features)[0]

print("Classe prevista:", pred_final)

In [ ]:
media_amostra = float(np.mean(X_test[indice]))
desvio_amostra = float(np.std(X_test[indice]))
max_amostra = float(np.max(X_test[indice]))
min_amostra = float(np.min(X_test[indice]))

print(media_amostra)
print(desvio_amostra)
print(max_amostra)
print(min_amostra)

In [39]:
%pip install -U -q google-genai

Note: you may need to restart the kernel to use updated packages.


In [40]:
from getpass import getpass
from google import genai

client = genai.Client(api_key=getpass("Cole sua chave do Google Gemini: "))

In [41]:
from getpass import getpass
from google import genai

client = genai.Client(api_key=getpass("Cole sua chave do Google Gemini: "))

prompt = f"""
Um modelo de inteligência artificial analisou um trecho de EEG.

Resumo da amostra:
- média: {media_amostra:.6f}
- desvio padrão: {desvio_amostra:.6f} 
- valor máximo amostra: {max_amostra:.6f}
- valor mínimo amostra: {min_amostra:.6f}

Classe prevista: {pred_final}

Explique de forma simples o que isso significa.
"""

response = client.interactions.create(
    model="gemini-3.6-flash",
    input=prompt
)

print("\nExplicação da IA:\n")
print(response.output_text)


Explicação da IA:

De forma bem simples, o resultado significa o seguinte:

Uma Inteligência Artificial olhou para um **pequeno trecho do sinal elétrico do cérebro** (o EEG) e concluiu que ele pertence ao grupo chamado **"Classe 0"**.

Aqui está o detalhamento do que cada parte significa:

---

### 1. O Resultado Principal: "Classe prevista: 0"
Em modelos de IA médica, as "classes" são as categorias que a máquina aprendeu a identificar. Na maioria dos estudos de EEG:
* **Classe 0** costuma significar o **estado normal, neutro ou de repouso** (por exemplo: *"sem crise epiléptica"*, *"paciente acordado/saudável"* ou *"sem anomalias"*).
* **Classe 1** (ou outras) costuma ser usada para identificar um evento específico (como uma crise, um sinal de sono profundo ou uma alteração).

👉 **Resumo:** A IA analisou esse trecho e entendeu que ele se encaixa no padrão padrão/normal (Classe 0) definido pelo seu criador.

---

### 2. O que os números significam?
Os números são muito pequenos (cheios